# 02 · A reproducible residential sale cohort

Cleaning defines the population the model can estimate. This notebook uses the same module as the training command so rules cannot silently diverge.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.figsize': (10, 4)})

from src.preprocessing import clean_transactions, SALE_PROCEDURES
raw = pd.read_csv(ROOT / 'data/raw/transactions-2026-09-11.csv')
print('Included procedures:', sorted(SALE_PROCEDURES))

Included procedures: ['Delayed Sell', 'Sale', 'Sale On Payment Plan', 'Sell - Pre registration']


## Eligibility and invalid values
Keep Sale, Sell - Pre registration, Delayed Sell and Sale On Payment Plan as an explicit working definition of individual sales. Exclude development, land-addition and lease-to-own procedures pending domain review. This does not prove each retained row is an individual sale. Reject missing/invalid dates, nonpositive or nonfinite prices and surface, duplicate transaction IDs, and conflicting price or feature records for the same ID. V2 also rejects explicit Office/Shop room labels on residential subtypes; one February record is affected. Quarantine known area ratios outside [0.5, 2]; keep missing procedure area because actual surface is available. This threshold is a scope heuristic, not an IQR price filter.

In [2]:
clean, audit = clean_transactions(raw)
display(pd.Series(audit, name='rows'))
display(clean.head())
print('Date range:', clean.INSTANCE_DATE.min(), clean.INSTANCE_DATE.max())
display(clean.isna().mean().sort_values(ascending=False).to_frame('missing_fraction'))

input_rows                       113866
outside_residential_sales         12865
excluded_procedure                 2745
invalid_date_price_or_surface         0
area_scope_mismatch                 249
commercial_room_mismatch              1
conflicting_transaction_id            0
duplicate_transaction_id             19
output_rows                       97987
Name: rows, dtype: int64

,ACTUAL_AREA,INSTANCE_DATE,AREA_EN,PROP_SB_TYPE_EN,ROOMS_EN,IS_OFFPLAN_EN,IS_FREE_HOLD_EN,PROJECT_EN,TRANS_VALUE
0,72.41,2026-01-01 16:21:33,Jabal Ali First,Flat,1 B/R,Off-Plan,Free Hold,Hills View at Wasl Gate,1099573.00
1,38.69,2026-01-01 19:43:16,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights,720622.22
2,38.69,2026-01-01 19:43:29,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights,716711.12
3,38.54,2026-01-01 19:43:32,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights,713777.79
4,77.64,2026-01-01 19:43:35,JUMEIRAH HEIGHTS,Flat,1 B/R,Off-Plan,Free Hold,ELTIERA HEIGHTS,2097828.00


Date range: 2026-01-01 16:21:33 2026-09-11 11:44:44


,missing_fraction
PROJECT_EN,0.054344
ROOMS_EN,0.003786
ACTUAL_AREA,0.000000
INSTANCE_DATE,0.000000
AREA_EN,0.000000
PROP_SB_TYPE_EN,0.000000
IS_OFFPLAN_EN,0.000000
IS_FREE_HOLD_EN,0.000000
TRANS_VALUE,0.000000


## Feature engineering and missing values
**Feature engineering** turns raw fields into model inputs. Date produces year, month and quarter; day-of-week is omitted because administrative recording schedules need not describe property value. Missing categories become Unknown inside the pipeline. PROJECT_EN and AREA_EN use rare-category buckets and bounded one-hot encoding fitted only on training data. PARKING is excluded because bay identifiers such as B-99 are not counts. PROCEDURE_AREA is excluded because it can have a different scope from ACTUAL_AREA. No target statistics are used to impute features.

In [3]:
from src.features import FeatureBuilder, INPUT_COLUMNS, FORBIDDEN
features = FeatureBuilder().fit_transform(clean[INPUT_COLUMNS])
display(features.head())
assert not set(features.columns) & FORBIDDEN

,ACTUAL_AREA,LOG_AREA,YEAR,MONTH,QUARTER,AREA_EN,PROP_SB_TYPE_EN,ROOMS_EN,IS_OFFPLAN_EN,IS_FREE_HOLD_EN,PROJECT_EN
0,72.41,4.282344,2026,1,1,Jabal Ali First,Flat,1 B/R,Off-Plan,Free Hold,Hills View at Wasl Gate
1,38.69,3.655581,2026,1,1,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights
2,38.69,3.655581,2026,1,1,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights
3,38.54,3.651697,2026,1,1,DUBAI LAND RESIDENCE COMPLEX,Flat,Studio,Off-Plan,Free Hold,Samana Boulevard Heights
4,77.64,4.352083,2026,1,1,JUMEIRAH HEIGHTS,Flat,1 B/R,Off-Plan,Free Hold,ELTIERA HEIGHTS


## Review the impact of the ratio rule
The [0.5, 2] bounds are conservative and may remove legitimate transactions. Compare narrower and wider candidate bounds before future iterations, using development data only for model selection. The counts below are an audit, not evidence that a threshold optimizes price prediction.

In [4]:
eligible = raw.loc[raw.PROCEDURE_EN.isin(SALE_PROCEDURES) & raw.PROP_SB_TYPE_EN.isin(__import__('src.preprocessing', fromlist=['RESIDENTIAL_TYPES']).RESIDENTIAL_TYPES)]
ratio = eligible.PROCEDURE_AREA / eligible.ACTUAL_AREA
display(pd.DataFrame([{'bounds':str((lo,hi)), 'flagged_rows':int((ratio.notna() & ~ratio.between(lo,hi)).sum())} for lo,hi in [(.25,4),(.5,2),(.8,1.25)]]))

,bounds,flagged_rows
0,"(0.25, 4)",66
1,"(0.5, 2)",249
2,"(0.8, 1.25)",566


## Save deterministic output
The processed CSV contains the target plus the inference schema. It contains no price-per-square-metre diagnostic. Imputation and encoding remain inside the model pipeline to avoid learning from the test set.

In [5]:
import json
out = ROOT / 'data/processed'
out.mkdir(parents=True, exist_ok=True)
clean.to_csv(out / 'residential_sales.csv', index=False)
(out / 'cleaning_audit.json').write_text(json.dumps(audit, indent=2))
print('Saved', len(clean), 'rows')

Saved 97987 rows
